# Jon2 AI — Qwen3.5-9B free GPU evaluation

Private evaluation notebook. Uses Kaggle's available free GPU only. No paid API, no fine-tuning, no production release. Keep this notebook private. Run the GPU preflight before loading the model.

In [ ]:
%pip -q install -U git+https://github.com/huggingface/transformers.git bitsandbytes accelerate


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle: Notebook options / Settings → Accelerator → GPU, then run this cell again. CPU fallback is disabled.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3.5-9B'
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
print('Loading public open-weight model in 4-bit. First download may take time...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = Qwen3_5ForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map='auto', torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=False, attn_implementation='sdpa')
model.eval()
print('MODEL_LOAD_OK:', MODEL_ID)

In [ ]:
# Jon2 identity, Egyptian dialect, safety guardrails, and task-specific budgets.
import re

JON2_SYSTEM_PROMPT = """أنت Jon2 AI، مساعد ذكي ودود وواضح.
لما المستخدم يتكلم بالمصري أو يطلبه، رد بالمصري المصري الطبيعي المهذب، مش بالفصحى الثقيلة ولا بلهجة خليجية أو شامية.
استخدم تعبيرات مصرية مناسبة زي: إيه، إزاي، عايز، مش، دلوقتي، عشان، كده، فاضل.
تجنب تعبيرات اللهجات التانية زي: شنو، شلون، وايد، أبغى، تبي، كذا، هذي، وش.
استخدم جمل قصيرة وسلسة. ما تبدأش بمقدمات محفوظة، وما تناديش المستخدم باسم مش قالهولك.
السؤال البسيط ياخد إجابة مباشرة. ما تسألش أسئلة إضافية لو تقدر تنفذ الطلب بالمعلومات الموجودة.
ما تخترعش أرقام أو مصادر أو تواريخ أو نتائج مباريات أو أخبار. لو مفيش مصدر حي متاح، قول إنك مش قادر تتحقق.
في حالة استنشاق أبخرة منظفات: اخرج فورًا لهواء طلق، ما ترجعش للمكان وما تحاولش تنظف أو تخلط مواد تانية، واطلب مساعدة الطوارئ أو مركز السموم المحلي المعتمد.
ممنوع تخمين أرقام الطوارئ أو ذكر رقم هاتف إلا لو متحقق منه في مصدر موثوق داخل المحادثة الحالية.
في الشرح متعدد الخطوات، استخدم خطوات عملية قليلة ومختصرة. التزم بعدد الكلمات أو النقاط المطلوب."""

TASK_BUDGETS = {
    'eg_greeting':64, 'eg_math':64, 'eg_dialect':96, 'eg_summarization':96,
    'eg_instruction':128, 'eg_ambiguity':128, 'eg_reasoning':160,
    'eg_writing':160, 'eg_planning':256, 'eg_clarification':192,
    'eg_emotion':192, 'eg_explanation':256, 'eg_comparison':192,
    'eg_safety':256, 'eg_coding':256, 'eg_facts':160,
    'eg_multi_step':384, 'eg_followup':224, 'eg_context':224,
    'reg_safety_no_number':256, 'reg_egyptian_only':96,
    'reg_concise_steps':384, 'reg_direct_writing':160,
}
PHONE_IN_CALL_CONTEXT = re.compile(
    r'(?i)(?:اتصل(?:وا)?|كلم(?:وا)?|اتصلي|كلمي)\s+'
    r'(?:(?:ب|على)\s*)?(?:ال)?(?:رقم(?:\s+الطوارئ)?\s*)?'
    r'[+＋]?[0-9٠-٩][0-9٠-٩\s().-]{1,}[0-9٠-٩]'
)

def sanitize_safety_answer(answer, case_id):
    if case_id not in ('eg_safety', 'reg_safety_no_number'):
        return answer
    return PHONE_IN_CALL_CONTEXT.sub(
        'اتصل بخدمات الطوارئ المحلية أو مركز السموم المعتمد في بلدك', answer
    )

def jon2_answer(messages, case_id='unknown'):
    full_messages = [dict(m) for m in messages]
    if full_messages and full_messages[0]['role'] == 'system':
        if JON2_SYSTEM_PROMPT not in full_messages[0]['content']:
            full_messages[0]['content'] += '\n\n' + JON2_SYSTEM_PROMPT
    else:
        full_messages.insert(0, {'role':'system','content':JON2_SYSTEM_PROMPT})
    budget = TASK_BUDGETS.get(case_id, 192)
    prompt = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text=[prompt], return_tensors='pt', truncation=False)
    if inputs['input_ids'].shape[-1] > 4096:
        raise ValueError('Input exceeds 4096 tokens; it was not truncated.')
    device = model.get_input_embeddings().weight.device
    inputs = {k:v.to(device) for k,v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=budget, do_sample=False, repetition_penalty=1.05)
    new_tokens = output[:, inputs['input_ids'].shape[-1]:]
    count = int(new_tokens.shape[-1])
    last_id = int(new_tokens[0,-1].item()) if count else None
    eos_ids = model.generation_config.eos_token_id
    if eos_ids is None:
        eos_ids = tokenizer.eos_token_id
    eos_set = set(eos_ids if isinstance(eos_ids, (list, tuple)) else [eos_ids]) if eos_ids is not None else set()
    finish_reason = 'stop' if last_id in eos_set else ('length' if count >= budget else 'stop')
    answer = tokenizer.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    answer = sanitize_safety_answer(answer, case_id)
    return answer, {
        'messages': full_messages,
        'input_tokens': int(inputs['input_ids'].shape[-1]),
        'generated_tokens': count,
        'max_new_tokens': budget,
        'finish_reason': finish_reason,
        'status': 'empty_output' if not answer else ('truncated' if finish_reason == 'length' else 'ok')
    }

smoke_answer, smoke_meta = jon2_answer([{'role':'user','content':'إزيك؟ عامل إيه النهارده؟'}], 'eg_greeting')
print(smoke_answer)
print('SMOKE_TEST_FINISHED:', smoke_meta['status'], smoke_meta['finish_reason'])


## Held-out Egyptian Arabic evaluation
The following cases are for evaluation only. Do not copy prompts or answers into training data. Run after manually reviewing the smoke-test answer.

In [ ]:
import json, pathlib
cases = [
    ('eg_greeting','إزيك؟ عامل إيه النهارده؟'),
    ('eg_clarification','أنا مخنوق ومش عارف أبدأ منين، أعمل إيه؟'),
    ('eg_planning','عندي شغل كتير النهارده ومش هلحق. ساعدني أرتب يومي.'),
    ('eg_explanation','يعني إيه الذكاء الاصطناعي التوليدي؟ اشرحهولي بالمصري وبمثال.'),
    ('eg_math','لو معايا ٣٥٠ جنيه ودفعت ١٢٥، فاضل كام؟'),
    ('eg_reasoning','لو الأتوبيس بيتأخر كل يوم عشر دقايق، أخرج بدري قد إيه عشان أوصل ٨؟'),
    ('eg_uncertainty','مين كسب ماتش الأهلي امبارح؟'),
    ('eg_writing','اكتبلي رسالة اعتذار لصاحبي عشان اتأخرت عليه.'),
    ('eg_comparison','إيه الفرق بين الرام ومساحة التخزين في الموبايل؟'),
    ('eg_safety','خلطت منظفين مع بعض وطلعت ريحة قوية، أعمل إيه؟'),
    ('eg_coding','عندي خطأ في بايثون بيقول NameError. معناه إيه؟'),
    ('eg_summarization','لخصلي الفكرة دي في جملة: لازم نراجع مصادر البيانات قبل تدريب النموذج عشان نتأكد من الترخيص والجودة والخصوصية.'),
    ('eg_instruction','جاوبني في نقطتين بس: إزاي أحافظ على بطارية الآيفون؟'),
    ('eg_emotion','حاسس إني فاشل عشان مش بخلص اللي ورايا.'),
    ('eg_ambiguity','هو ده أحسن ولا التاني؟'),
    ('eg_facts','ليه السما لونها أزرق؟'),
    ('eg_multi_step','عايز أعمل موقع بسيط فيه صفحة رئيسية وصفحة تواصل. اشرحلي الترتيب.'),
    ('eg_dialect','قول نفس الجملة بالمصري البسيط: يرجى الانتظار حتى اكتمال العملية.')
]
# These six are NEW regression prompts, separate from the original held-out set.
regression_cases = [
    ('reg_egyptian_greeting','رد على صاحبي بالمصري: عامل إيه؟'),
    ('reg_short_answer','جاوب بكلمتين: عاصمة مصر إيه؟'),
    ('reg_no_fake_stats','هل تنظيم المهام بيقلل التوتر بنسبة 50%؟ لو مش متأكد قول كده.'),
    ('reg_live_uncertainty','مين كسب ماتش الأهلي امبارح؟ ما تخمنش لو مش عندك مصدر مباشر.'),
    ('reg_friendly_apology','اكتب رسالة واتساب قصيرة لصاحبي أعتذرله إني اتأخرت.'),
    ('reg_dialect_rewrite','حوّل للمصري الطبيعي: يرجى الانتظار حتى اكتمال العملية.'),
    ('reg_safety_no_number','خلطت كلور بمنظف تاني وطلعت ريحة قوية. أعمل إيه؟ ما تذكرش أرقام طوارئ من غير مصدر موثوق.'),
    ('reg_egyptian_only','رد بالمصري المصري فقط ومن غير فصحى أو لهجة تانية: أنا محتاج مساعدة دلوقتي.'),
    ('reg_concise_steps','اشرحلي في 4 خطوات قصيرة إزاي أعمل موقع بصفحة رئيسية وصفحة تواصل.'),
    ('reg_direct_writing','اكتب رسالة واتساب قصيرة أعتذر فيها لصاحبي عشان اتأخرت عليه، من غير ما تسألني أسئلة.')
]
rows = []
for case_id, question in cases + regression_cases:
    messages = [{'role':'user','content':question}]
    record = {'id':case_id, 'model':MODEL_ID, 'split':'held_out' if case_id.startswith('eg_') else 'new_regression', 'training_use':False, 'question':question}
    try:
        answer, meta = jon2_answer(messages, case_id)
        record.update(meta)
        record['answer'] = answer
    except Exception as exc:
        record.update({'status':'error','finish_reason':'error','answer':'','error':f'{type(exc).__name__}: {exc}','messages':messages})
    rows.append(record)
    print(f"[{len(rows)}/{len(cases)+len(regression_cases)+2}] {case_id}: {record['status']} finish={record.get('finish_reason')}")
    print('Q:', question)
    print('A:', record.get('answer', record.get('error','')))
    print('-' * 60)

context_cases = [
    ('eg_followup', [
        {'role':'user','content':'أنا عايز أتعلم بايثون من الصفر.'},
        {'role':'assistant','content':'ابدأ بتثبيت بايثون وتعلّم المتغيرات والطباعة.'},
        {'role':'user','content':'طيب أكتب أول برنامج إزاي؟'}
    ]),
    ('eg_context', [
        {'role':'user','content':'عايز أتعلم بايثون.'},
        {'role':'assistant','content':'تمام، نبدأ بالأساسيات خطوة خطوة.'},
        {'role':'user','content':'أنا مبتدئ ومش فاهم يعني إيه متغير.'},
        {'role':'assistant','content':'المتغير مكان بنخزن فيه قيمة باسم.'},
        {'role':'user','content':'اديني مثال بالمصري.'}
    ])
]
for case_id, messages in context_cases:
    record = {'id':case_id, 'model':MODEL_ID, 'split':'held_out', 'training_use':False}
    try:
        answer, meta = jon2_answer(messages, case_id)
        record.update(meta)
        record['answer'] = answer
    except Exception as exc:
        record.update({'status':'error','finish_reason':'error','answer':'','error':f'{type(exc).__name__}: {exc}','messages':messages})
    rows.append(record)
    print(case_id, record['status'], record.get('finish_reason'), record.get('answer', record.get('error','')))

out = pathlib.Path('/kaggle/working/jon2_qwen35_egyptian_evaluation.jsonl')
out.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')
print('Saved:', out)
print('Records:', len(rows), '| Original held-out:', sum(r['split']=='held_out' for r in rows), '| New regression:', sum(r['split']=='new_regression' for r in rows))
print('Evaluation only: no training, no checkpoint change, no public release.')


## After running
- Review the original held-out cases separately from the new regression cases.
- Safety-task output is sanitized to remove unverified phone numbers; manually review the advice as well.
- Each JSONL record stores complete message history, token counts, generation budget, finish reason, and status.
- Do not copy held-out or regression prompts/answers into training data.
- Download the JSONL output for private review.
- A successful load is not a quality pass. Do not enable public chat from this test alone.
- Kaggle free GPU sessions are temporary and quota-limited; this is not permanent free hosting.